# Graph Features Exploration

**Goal:** Characterize current graph morphometry features, run QC and sanity checks, investigate batch/site effects and predictive signal, and deliver a concise conclusion on likely causes of weak pCR prediction signal.

---

## 1. Setup and config

Paths, constants, and random seed for reproducible runs. **NOTE:** These assume that this notebook is located in the DSI Cluster!

In [16]:
import json
import logging
from pathlib import Path

import numpy as np
import pandas as pd

In [17]:
# ---------------------------------------------------------------------------
# Paths
# ---------------------------------------------------------------------------
BASE = Path("/net/projects2/vanguard")

# Dataset Directory
DATASET_DIR = BASE / "MAMA-MIA-syn60868042"

# Directory of per-case morphometry JSONs (one file per case; see ML-Pipeline/pcr_prediction.py)
FEATURE_DIR = DATASET_DIR / "patient_info_files"

# Metadata with patient_id and site/scanner/cohort
METADATA_PATH = DATASET_DIR / "clinical_and_imaging_info.xlsx"

# Labels (pCR): CSV or directory of label JSONs
LABELS_PATH = (
    FEATURE_DIR / "pcr_labels.csv"
)  # e.g. Path("pcr_labels.csv") or BASE / "labels"

# Train/val splits and site proxy (patient_id, fold_idx, dataset, stratum_key)
SPLITS_CSV = Path("splits.csv")

# ---------------------------------------------------------------------------
# Constants
# ---------------------------------------------------------------------------
EXPECTED_N_CASES = 1500  # approximate cohort size for sanity checks
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

In [18]:
# Print config and basic path checks
print("Config:")
print(f"  FEATURE_DIR      = {FEATURE_DIR}")
print(f"  METADATA_PATH    = {METADATA_PATH}")
print(f"  LABELS_PATH      = {LABELS_PATH}")
print(f"  SPLITS_CSV       = {SPLITS_CSV}")
print(f"  EXPECTED_N_CASES = {EXPECTED_N_CASES}")
print(f"  RANDOM_STATE     = {RANDOM_STATE}")
print()

if FEATURE_DIR.exists():
    n_jsons = len(list(FEATURE_DIR.glob("*.json")))
    print(f"  FEATURE_DIR exists: {n_jsons} .json files")
else:
    print(
        "  FEATURE_DIR does not exist (will need to set or use pre-built features.csv)"
    )
if SPLITS_CSV.exists():
    print(f"  SPLITS_CSV exists: {len(pd.read_csv(SPLITS_CSV))} rows")
else:
    print("  SPLITS_CSV not found")

Config:
  FEATURE_DIR      = /net/projects2/vanguard/MAMA-MIA-syn60868042/patient_info_files
  METADATA_PATH    = /net/projects2/vanguard/MAMA-MIA-syn60868042/clinical_and_imaging_info.xlsx
  LABELS_PATH      = /net/projects2/vanguard/MAMA-MIA-syn60868042/patient_info_files/pcr_labels.csv
  SPLITS_CSV       = splits.csv
  EXPECTED_N_CASES = 1500
  RANDOM_STATE     = 42

  FEATURE_DIR exists: 1506 .json files
  SPLITS_CSV exists: 1480 rows


## 2. Data loading

Load the feature matrix (from per-case morphometry JSONs or a pre-built CSV), clinical/imaging metadata from `clinical_and_imaging_info.xlsx`, and optional splits. Align everything by `patient_id` (metadata) / `case_id` (features; derived from JSON filename as first two underscore-separated parts, e.g. `DUKE_001`).

In [19]:
# ---------------------------------------------------------------------------
# Load clinical and imaging metadata (clinical_and_imaging_info.xlsx)
# Fields: patient_id, dataset, site, manufacturer, scanner_model, field_strength,
#         pcr, tumor_subtype, age, and others (see data dictionary).
# ---------------------------------------------------------------------------
if METADATA_PATH is not None and METADATA_PATH.exists():
    meta = pd.read_excel(METADATA_PATH, sheet_name=0, engine="openpyxl")
    # Normalize column names (strip whitespace)
    meta.columns = meta.columns.str.strip()
    # Key columns for downstream: id, outcome, batch/site, stratification
    id_col = "patient_id"
    label_col = "pcr"
    site_cols = ["dataset", "site", "manufacturer", "scanner_model", "field_strength"]
    strat_col = "tumor_subtype"
    for c in [id_col, label_col]:
        if c not in meta.columns:
            raise KeyError(f"Metadata missing required column: {c}")
    # Coerce pcr to numeric (0/1); empty or non-numeric -> NaN
    meta[label_col] = pd.to_numeric(meta[label_col], errors="coerce")
    print(f"Metadata: {len(meta)} rows, {len(meta.columns)} columns")
    print(f"  pCR non-null: {meta[label_col].notna().sum()}")
    print(
        f"  Site-related columns present: {[c for c in site_cols if c in meta.columns]}"
    )
else:
    meta = None
    id_col, label_col, site_cols, strat_col = "patient_id", "pcr", [], "tumor_subtype"
    print("METADATA_PATH not set or file missing; skipping metadata load.")

Metadata: 1506 rows, 50 columns
  pCR non-null: 1491
  Site-related columns present: ['dataset', 'site', 'manufacturer', 'scanner_model', 'field_strength']


In [20]:
# ---------------------------------------------------------------------------
# Load feature matrix: from pre-built CSV or build from per-case JSONs
# ---------------------------------------------------------------------------
FEATURES_CSV = None  # Set to Path("features.csv") if you have a pre-built table


def build_features_from_jsons(feature_dir: Path) -> pd.DataFrame:
    """Build one row per case from morphometry JSONs (matches ML-Pipeline/pcr_prediction logic)."""
    rows = []
    for p in sorted(feature_dir.glob("*.json")):
        try:
            data = json.loads(p.read_text())
        except Exception as e:
            logging.warning("Skipping %s: %s", p, e)
            continue
        fname = p.stem
        case_id = "_".join(fname.split("_")[:2])
        feats = {"case_id": case_id}

        def to_num(x: object) -> float | None:
            if isinstance(x, bool | np.bool_):
                return float(x)
            if isinstance(x, int | float | np.integer | np.floating):
                return float(x)
            if isinstance(x, str):
                try:
                    return float(x.strip())
                except Exception:
                    return None
            return None

        if not isinstance(data, dict):
            rows.append(feats)
            continue
        for _, group in data.items():
            if not isinstance(group, dict):
                continue
            for vessel_name, items in group.items():
                if not isinstance(items, list):
                    continue
                per_item_vals = {}
                for item in items:
                    if not isinstance(item, dict):
                        continue
                    for k, v in item.items():
                        if isinstance(v, dict):
                            for kk, vv in v.items():
                                if isinstance(vv, list):
                                    nums = [
                                        to_num(t) for t in vv if to_num(t) is not None
                                    ]
                                    if nums:
                                        per_item_vals.setdefault(
                                            f"{k}__{kk}", []
                                        ).extend(nums)
                                else:
                                    n = to_num(vv)
                                    if n is not None:
                                        per_item_vals.setdefault(
                                            f"{k}__{kk}", []
                                        ).append(n)
                        elif isinstance(v, list):
                            nums = [to_num(t) for t in v if to_num(t) is not None]
                            if nums:
                                per_item_vals.setdefault(k, []).extend(nums)
                        else:
                            n = to_num(v)
                            if n is not None:
                                per_item_vals.setdefault(k, []).append(n)
                vprefix = vessel_name.replace(" ", "_")
                for fld, vals in per_item_vals.items():
                    if not vals:
                        continue
                    base = f"{vprefix}__{fld}"
                    feats[f"{base}__mean"] = float(np.mean(vals))
                    feats[f"{base}__std"] = (
                        float(np.std(vals)) if len(vals) > 1 else 0.0
                    )
                    feats[f"{base}__min"] = float(np.min(vals))
                    feats[f"{base}__max"] = float(np.max(vals))
                    feats[f"{base}__count"] = float(len(vals))
        rows.append(feats)
    features_raw = pd.DataFrame(rows)
    num = features_raw.select_dtypes(include=["number"]).copy()
    if "case_id" in features_raw.columns and "case_id" not in num.columns:
        num = pd.concat([features_raw["case_id"], num], axis=1)
    num = num.dropna(axis=1, how="all")
    cols_no_id = [c for c in num.columns if c != "case_id"]
    if cols_no_id:
        const = [c for c in cols_no_id if num[c].eq(num[c].iloc[0]).all()]
        num = num.drop(columns=const, errors="ignore")
    return num


if FEATURES_CSV is not None and Path(FEATURES_CSV).exists():
    features_df = pd.read_csv(FEATURES_CSV)
    if "case_id" not in features_df.columns and "patient_id" in features_df.columns:
        features_df = features_df.rename(columns={"patient_id": "case_id"})
    print(
        f"Features: loaded from {FEATURES_CSV} -> {len(features_df)} rows, {len(features_df.columns)} columns"
    )
elif FEATURE_DIR.exists():
    features_df = build_features_from_jsons(FEATURE_DIR)
    print(
        f"Features: built from JSONs in {FEATURE_DIR} -> {len(features_df)} rows, {len(features_df.columns)} columns"
    )
else:
    features_df = pd.DataFrame(columns=["case_id"])
    print("No feature directory or pre-built CSV; features_df is empty.")

KeyboardInterrupt: 

In [ ]:
# ---------------------------------------------------------------------------
# Merge features with metadata (case_id = patient_id) and optional splits
# ---------------------------------------------------------------------------
if meta is not None and len(features_df) > 0 and "case_id" in features_df.columns:
    merged = features_df.merge(
        meta[
            [c for c in [id_col, label_col, strat_col] + site_cols if c in meta.columns]
        ],
        left_on="case_id",
        right_on=id_col,
        how="inner",
    )
    if SPLITS_CSV.exists():
        splits_df = pd.read_csv(SPLITS_CSV)
        # Add only fold_idx and stratum_key (dataset already in metadata)
        splits_sub = splits_df[["patient_id", "fold_idx", "stratum_key"]]
        merged = merged.merge(
            splits_sub,
            on=id_col,
            how="left",
        )
    print(f"Merged (features + metadata): {len(merged)} rows")
    print(f"  With non-missing pCR:     {merged[label_col].notna().sum()}")
    if "dataset" in merged.columns:
        print(
            f"  dataset value_counts:\n{merged['dataset'].value_counts().to_string()}"
        )
else:
    merged = features_df.copy() if len(features_df) > 0 else pd.DataFrame()
    if meta is not None and "patient_id" in meta.columns:
        merged = (
            merged.merge(meta, left_on="case_id", right_on="patient_id", how="left")
            if "case_id" in merged.columns
            else meta
        )
    print("Merged: no merge performed (missing features or metadata).")

Merged (features + metadata): 0 rows
  With non-missing pCR:     0
  dataset value_counts:
Series([], )


In [ ]:
# Summary: sample sizes for downstream sections
n_cases = len(merged)
skip_cols = {id_col, "case_id", label_col, strat_col, "fold_idx"} | set(site_cols)
feature_cols = [
    c
    for c in merged.columns
    if c not in skip_cols and pd.api.types.is_numeric_dtype(merged[c])
]
n_with_label = merged[label_col].notna().sum()

print(f"Cases in merged table: {n_cases}")
print(f"Cases with non-missing pCR: {n_with_label}")
print(f"Numeric feature columns (excl. id/label/site): {len(feature_cols)}")

Cases in merged table: 0
Cases with non-missing pCR: 0
Numeric feature columns (excl. id/label/site): 0


## 3. Human-readable feature list

Graph morphometry features are produced by the skeleton→graph→morphometry pipeline ([skeleton_to_graph.py](graph_pruning_centerline_extraction/skeleton3d_utils/skeleton_to_graph.py), [centerline_to_json.py](batch_processing/centerline_to_json.py)). Each **segment** yields: radius stats, length, tortuosity, volume, curvature stats; **bifurcations** yield opening angles. The ML pipeline flattens these per vessel/component into one row per case with suffixes `__mean`, `__std`, `__min`, `__max`, `__count`. Below we group the actual columns in this run by feature type and list them for reference.

In [ ]:
# Feature type definitions (from pipeline schema)
FEATURE_TYPE_DEFINITIONS = {
    "radius": "Vessel radius at skeleton points (from distance transform); stats: mean, sd, median, min, q1, q3, max. Units: voxels or mm if spacing applied. Expect > 0.",
    "length": "Path length of vessel segment (sum of edge lengths). Units: voxels or mm. Expect > 0.",
    "tortuosity": "Path length / straight-line distance; ≥ 1. (Some pipelines use (path/straight)−1.)",
    "curvature": "Angular change per unit path (rad or deg); stats as above. Expect ≥ 0.",
    "angle": "Bifurcation opening angles between branch pairs. Expect [0, 180] degrees.",
    "volume": "Segment volume (e.g. π·r²·length or sum of voxel volumes). Expect > 0.",
    "count": "Number of segments or points contributing to the aggregate (__count suffix).",
    "other": "Other numeric fields from the JSON (e.g. segment endpoints, coordinates).",
}

In [ ]:
# Map keywords (lowercase) to feature type for grouping
KEYWORD_TO_TYPE = {
    "radius": "radius",
    "length": "length",
    "tortuosity": "tortuosity",
    "curvature": "curvature",
    "angle": "angle",
    "volume": "volume",
    "area": "volume",  # treat area as volume-like
    "count": "count",
}


def classify_feature_column(name: str) -> str:
    """Return feature type (radius, length, tortuosity, etc.) from column name."""
    n = name.lower()
    for kw, ftype in KEYWORD_TO_TYPE.items():
        if kw in n:
            return ftype
    return "other"


# Build grouped list from actual feature columns
if feature_cols:
    grouped = {}
    for c in feature_cols:
        ftype = classify_feature_column(c)
        grouped.setdefault(ftype, []).append(c)
    for ftype in [
        "radius",
        "length",
        "tortuosity",
        "curvature",
        "angle",
        "volume",
        "count",
        "other",
    ]:
        if ftype in grouped:
            grouped[ftype] = sorted(grouped[ftype])
    feature_groups = grouped
else:
    feature_groups = {}

In [ ]:
# Display: definitions table + column counts and sample column names per type
print("Feature type definitions (from pipeline)\n")
for ftype, desc in FEATURE_TYPE_DEFINITIONS.items():
    print(f"  {ftype}: {desc}\n")

print("\nColumn counts and example columns per type (this run):")
for ftype in [
    "radius",
    "length",
    "tortuosity",
    "curvature",
    "angle",
    "volume",
    "count",
    "other",
]:
    cols = feature_groups.get(ftype, [])
    n = len(cols)
    examples = cols[:3] if n else []
    print(f"  {ftype}: {n} columns  e.g. {examples}")

Feature type definitions (from pipeline)

  radius: Vessel radius at skeleton points (from distance transform); stats: mean, sd, median, min, q1, q3, max. Units: voxels or mm if spacing applied. Expect > 0.

  length: Path length of vessel segment (sum of edge lengths). Units: voxels or mm. Expect > 0.

  tortuosity: Path length / straight-line distance; ≥ 1. (Some pipelines use (path/straight)−1.)

  curvature: Angular change per unit path (rad or deg); stats as above. Expect ≥ 0.

  angle: Bifurcation opening angles between branch pairs. Expect [0, 180] degrees.

  volume: Segment volume (e.g. π·r²·length or sum of voxel volumes). Expect > 0.

  count: Number of segments or points contributing to the aggregate (__count suffix).

  other: Other numeric fields from the JSON (e.g. segment endpoints, coordinates).


Column counts and example columns per type (this run):
  radius: 0 columns  e.g. []
  length: 0 columns  e.g. []
  tortuosity: 0 columns  e.g. []
  curvature: 0 columns  e.g.

In [ ]:
# Optional: full list of feature column names (uncomment to inspect)
# for ftype, cols in feature_groups.items():
#     print(f"\n--- {ftype} ({len(cols)}) ---")
#     for c in cols:
#         print(f"  {c}")